# Step 1 — Legacy implementation

This notebook is retained for reproducibility. For new analyses, use `step1_corrected_output_folder.ipynb`.


In [ ]:
#1. Establishment of Spatial-Velo model using spatial in situ transcriptomic dataset
##requirements: tx file should conatain cell name, CellComp, gene name
##requirements: metadata file should contain cell name, position (x, y), cell boundary (min_x, min_y, max_x, max_y)

In [ ]:
# -*- coding: utf-8 -*-
"""
Integrated nuclear-export pipeline with selectable 2D / 3D r_norm.

Logic:
  1) Load transcripts
  2) Keep nuclear transcripts
  3) FIRST sample cells (shared by both 2D and 3D)
  4) Then compute nuclear diameters / 2D r_norm / 3D correction
  5) Fit per-cell PDE
  6) Optional spatial KS map
  7) Compute export probabilities, delta matrix, median-standardized delta, future matrix
"""

import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.optimize import minimize
from scipy.stats import kstest, gaussian_kde, pearsonr, spearmanr, linregress
from scipy.special import i0, i1
from scipy.interpolate import interp1d
from scipy.ndimage import gaussian_filter1d
from tqdm import tqdm

#------------------------------Path to tx and metadata file---------------------------------
_CWD_DATA_DIR = os.path.join(os.getcwd(), "data", "test_data")
_DEFAULT_WORKDIR = (
    _CWD_DATA_DIR
    if os.path.isdir(_CWD_DATA_DIR)
    else os.path.join("..", "data", "test_data")
)
BASE_DIR = os.path.abspath(
    os.environ.get("SPATIAL_VELO_WORKDIR", _DEFAULT_WORKDIR)
)
TX_IN_FILE = os.path.join(BASE_DIR, "xenium_test_tx_file.csv")
META_FILE = os.path.join(BASE_DIR, "xenium_test_metadata.csv")

#---------------------Below are parameters to change-------------------------------
#------------------Input tx file, transcript column names---------------------------
# Sampling
SAMPLE_SIZE = 10000
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

TX_CELL_COL = "cell"
TX_GENE_COL = "gene"
TX_COMP_COL = "CellComp"
TX_X_COL = "x"
TX_Y_COL = "y"

# Metadata column names for spatial map
META_CELL_COL = "cell"
META_X_COL = "x"
META_Y_COL = "y"

# Fallback/backup transcript column names
TX_CELL_FALLBACK = "cell_id"
TX_GENE_FALLBACK = "target"
TX_X_FALLBACK = "global_x"
TX_Y_FALLBACK = "global_x"

# Compartment labels in the CellComp column
NUCLEAR_LABEL = "nuclear"
CYTOPLASM_LABEL = "cytoplasm"

# Choose: "2D" or "3D"
NORM_MODE = os.environ.get("SPATIAL_VELO_NORM_MODE", "2D").upper()

#---------------------End of the list to change------------------------------

#Plot color limits: use 5% to 95% by default
PLOT_VMIN_PERCENTILE = 5
PLOT_VMAX_PERCENTILE = 95

OUT_ROOT = os.path.join(BASE_DIR, "pipeline_out")
OUT_DIR = os.path.join(OUT_ROOT, f"mode_{NORM_MODE}")
OUT_PART1 = os.path.join(OUT_DIR, "out_part1")
OUT_PART2 = os.path.join(OUT_DIR, "out_part2")
OUT_PART3 = os.path.join(OUT_DIR, "out_part3")
OUT_DESC = os.path.join(OUT_DIR, "descriptive")

for p in [OUT_PART1, OUT_PART2, OUT_PART3, OUT_DESC]:
    os.makedirs(p, exist_ok=True)

# Output files
CYTO_OUT_MATRIX = os.path.join(BASE_DIR, "transcripts_with_CellComp_nuc_matrix.csv")
DELTA_OUT_CSV = os.path.join(OUT_PART3, "delta_export_prob.csv")
DELTA_MEDIAN_OUT = os.path.join(OUT_PART3, "delta_export_prob_median.csv")
FUTURE_OUT = os.path.join(OUT_PART3, "future.csv")

# 3D correction config
DIAMETER_CSV_OUT = os.path.join(OUT_DESC, "nuclear_diameters.csv")
MAX_DIAMETER = None
DIAM_CELL_COL = "cell"
DIAMETER_COL = "diameter_px"


def standardize_transcripts(tx_df: pd.DataFrame, require_gene: bool = False) -> pd.DataFrame:
    """
    Make transcript table compatible with the pipeline.

    This function accepts either:
      - raw columns: TX_CELL_COL / TX_GENE_COL / TX_COMP_COL / TX_X_COL / TX_Y_COL
      - already standardized columns: cell / gene / CellComp / x / y
      - fallback raw columns: cell_ID / target / x / y
    """
    tx_df = tx_df.copy()

    # cell
    if "cell" not in tx_df.columns:
        if TX_CELL_COL in tx_df.columns:
            tx_df["cell"] = tx_df[TX_CELL_COL]
        elif TX_CELL_FALLBACK in tx_df.columns:
            tx_df["cell"] = tx_df[TX_CELL_FALLBACK]
        else:
            raise RuntimeError(
                f"Input transcripts file missing cell column. "
                f"Tried: 'cell', '{TX_CELL_COL}', '{TX_CELL_FALLBACK}'."
            )

    # CellComp
    if "CellComp" not in tx_df.columns:
        if TX_COMP_COL in tx_df.columns:
            tx_df["CellComp"] = tx_df[TX_COMP_COL]
        else:
            raise RuntimeError(
                f"Input transcripts file missing CellComp column. "
                f"Tried: 'CellComp', '{TX_COMP_COL}'."
            )

    # gene
    if "gene" not in tx_df.columns:
        if TX_GENE_COL in tx_df.columns:
            tx_df["gene"] = tx_df[TX_GENE_COL]
        elif TX_GENE_FALLBACK in tx_df.columns:
            tx_df["gene"] = tx_df[TX_GENE_FALLBACK]
        elif require_gene:
            raise RuntimeError(
                f"Input transcripts file missing gene column. "
                f"Tried: 'gene', '{TX_GENE_COL}', '{TX_GENE_FALLBACK}'."
            )

    # x
    if "x" not in tx_df.columns:
        if TX_X_COL in tx_df.columns:
            tx_df["x"] = tx_df[TX_X_COL]
        elif TX_X_FALLBACK in tx_df.columns:
            tx_df["x"] = tx_df[TX_X_FALLBACK]
        else:
            raise RuntimeError(
                f"Input transcripts file missing x coordinate column. "
                f"Tried: 'x', '{TX_X_COL}', '{TX_X_FALLBACK}'."
            )

    # y
    if "y" not in tx_df.columns:
        if TX_Y_COL in tx_df.columns:
            tx_df["y"] = tx_df[TX_Y_COL]
        elif TX_Y_FALLBACK in tx_df.columns:
            tx_df["y"] = tx_df[TX_Y_FALLBACK]
        else:
            raise RuntimeError(
                f"Input transcripts file missing y coordinate column. "
                f"Tried: 'y', '{TX_Y_COL}', '{TX_Y_FALLBACK}'."
            )

    tx_df["cell"] = tx_df["cell"].astype(str)
    tx_df["CellComp"] = tx_df["CellComp"].astype(str)
    if "gene" in tx_df.columns:
        tx_df["gene"] = tx_df["gene"].astype(str)

    tx_df["x"] = pd.to_numeric(tx_df["x"], errors="coerce")
    tx_df["y"] = pd.to_numeric(tx_df["y"], errors="coerce")
    tx_df = tx_df.dropna(subset=["x", "y"]).copy()

    return tx_df


def standardize_metadata(meta_df: pd.DataFrame) -> pd.DataFrame:
    """
    Rename configurable metadata columns to internal standard names:
    cell, meta_x, meta_y
    """
    required = [META_CELL_COL, META_X_COL, META_Y_COL]
    for col in required:
        if col not in meta_df.columns:
            raise RuntimeError(f"Metadata file missing required column: {col}")

    meta_df = meta_df.rename(columns={
        META_CELL_COL: "cell",
        META_X_COL: "meta_x",
        META_Y_COL: "meta_y",
    }).copy()

    meta_df["cell"] = meta_df["cell"].astype(str)
    meta_df["meta_x"] = pd.to_numeric(meta_df["meta_x"], errors="coerce")
    meta_df["meta_y"] = pd.to_numeric(meta_df["meta_y"], errors="coerce")
    meta_df = meta_df.dropna(subset=["meta_x", "meta_y"]).copy()

    return meta_df


def get_percentile_limits(values, low_pct=5, high_pct=95):
    arr = pd.to_numeric(pd.Series(values), errors="coerce").dropna().to_numpy(dtype=float)
    if len(arr) == 0:
        return 0.0, 1.0
    vmin = float(np.percentile(arr, low_pct))
    vmax = float(np.percentile(arr, high_pct))
    if np.isclose(vmin, vmax):
        vmax = vmin + 1e-12
    return vmin, vmax


# ============================================================
# STEP 0: load transcripts
# ============================================================
def load_transcripts(tx_file: str) -> pd.DataFrame:
    print("Step 0: Loading transcripts...")

    if not os.path.exists(tx_file):
        raise FileNotFoundError(f"Transcript file not found: {tx_file}")

    tx_df = pd.read_csv(tx_file, low_memory=False)
    print(f"  Loaded {len(tx_df)} transcript rows.")

    tx_df = standardize_transcripts(tx_df, require_gene=False)
    return tx_df


def get_nuclear_transcripts(tx_df: pd.DataFrame) -> pd.DataFrame:
    nuclear_tx = tx_df[tx_df["CellComp"].astype(str).str.lower() == NUCLEAR_LABEL].copy()
    if len(nuclear_tx) == 0:
        raise RuntimeError("No nuclear transcripts found.")
    print(f"  Nuclear transcripts: {len(nuclear_tx)}")
    print(f"  Cells with nuclear transcripts: {nuclear_tx['cell'].nunique()}")
    return nuclear_tx


def sample_cells_first(nuclear_tx: pd.DataFrame,
                       sample_size: int,
                       random_state: int = 42) -> pd.DataFrame:
    """
    Sample cells FIRST, then run all downstream 2D/3D analyses on the same subset.
    """
    cell_ids = nuclear_tx["cell"].dropna().astype(str).unique()
    total_cells = len(cell_ids)

    if sample_size is None or sample_size <= 0 or sample_size >= total_cells:
        print(f"  Sampling skipped: using all {total_cells} nuclear cells.")
        return nuclear_tx.copy()

    rng = np.random.RandomState(random_state)
    sampled_cells = rng.choice(cell_ids, size=sample_size, replace=False)

    sampled_tx = nuclear_tx[nuclear_tx["cell"].isin(sampled_cells)].copy()
    print(f"  Sampled cells: {len(sampled_cells)} / {total_cells}")
    print(f"  Nuclear transcripts after sampling: {len(sampled_tx)}")
    print(f"  Cells after sampling: {sampled_tx['cell'].nunique()}")

    return sampled_tx


# ============================================================
# STEP 1: compute nuclear diameters from sampled transcripts
# ============================================================
def compute_nuclear_diameters(tx_df: pd.DataFrame) -> pd.DataFrame:
    """
    Diameter = maximum Euclidean distance from nuclear transcripts to nuclear centroid
    Returns:
      cell, diameter_px, n_transcripts
    """
    print("Step 1: Computing nuclear diameters from sampled nuclear transcripts...")

    required_cols = ["CellComp", "x", "y", "cell"]
    for col in required_cols:
        if col not in tx_df.columns:
            raise RuntimeError(f"Missing required column: {col}")

    nuclear = tx_df[tx_df["CellComp"].astype(str).str.lower() == NUCLEAR_LABEL].copy()
    nuclear["x"] = pd.to_numeric(nuclear["x"], errors="coerce")
    nuclear["y"] = pd.to_numeric(nuclear["y"], errors="coerce")
    nuclear = nuclear.dropna(subset=["x", "y"]).copy()

    print(f"  Nuclear transcript rows used for diameter estimation: {len(nuclear)}")

    results = []
    for cell, group in nuclear.groupby("cell"):
        cx = group["x"].mean()
        cy = group["y"].mean()
        dists = np.sqrt((group["x"] - cx) ** 2 + (group["y"] - cy) ** 2)
        diameter = float(dists.max())
        results.append({
            "cell": str(cell),
            "diameter_px": diameter,
            "n_transcripts": int(len(group))
        })

    diam_df = pd.DataFrame(results)
    diam_df["cell"] = diam_df["cell"].astype(str)
    print(f"  Successfully computed diameters for {len(diam_df)} cells.")
    return diam_df


def plot_diameter_distribution(diam_df: pd.DataFrame, out_dir: str) -> None:
    """Plot diameter distribution, boxplot, scatter vs transcript count, and CDF"""
    if len(diam_df) == 0:
        print("  No diameter data to plot.")
        return

    sns.set_style("whitegrid")
    fig, axes = plt.subplots(2, 2, figsize=(12, 10))

    ax1 = axes[0, 0]
    sns.histplot(diam_df["diameter_px"], bins=50, kde=True, ax=ax1, color="steelblue")
    ax1.set_xlabel("Diameter (pixels)")
    ax1.set_ylabel("Number of cells")
    ax1.set_title("Distribution of nuclear diameters")

    ax2 = axes[0, 1]
    sns.boxplot(y=diam_df["diameter_px"], ax=ax2, color="lightcoral")
    ax2.set_ylabel("Diameter (pixels)")
    ax2.set_title("Boxplot of diameters")

    ax3 = axes[1, 0]
    ax3.scatter(diam_df["n_transcripts"], diam_df["diameter_px"], alpha=0.5, s=10, c="darkgreen")
    ax3.set_xlabel("Number of nuclear transcripts")
    ax3.set_ylabel("Diameter (pixels)")
    ax3.set_title("Diameter vs. nuclear transcript count")

    if len(diam_df) >= 2:
        slope, intercept, r_value, p_value, std_err = linregress(
            diam_df["n_transcripts"], diam_df["diameter_px"]
        )
        x_fit = np.linspace(diam_df["n_transcripts"].min(),
                            diam_df["n_transcripts"].max(), 100)
        y_fit = slope * x_fit + intercept
        ax3.plot(x_fit, y_fit, "r--", label=f"fit (R²={r_value**2:.2f})")
        ax3.legend()

    ax4 = axes[1, 1]
    sorted_diam = np.sort(diam_df["diameter_px"])
    cdf = np.arange(1, len(sorted_diam) + 1) / len(sorted_diam)
    ax4.plot(sorted_diam, cdf, marker=".", linestyle="none",
             markersize=2, color="purple")
    ax4.set_xlabel("Diameter (pixels)")
    ax4.set_ylabel("Cumulative fraction")
    ax4.set_title("Empirical CDF of diameters")

    plt.tight_layout()
    out_fig = os.path.join(out_dir, "nuclear_diameter_analysis.png")
    plt.savefig(out_fig, dpi=300)
    plt.close()
    print(f"  Diameter analysis plot saved: {out_fig}")


def print_statistics(diam_df: pd.DataFrame) -> None:
    print("\n========== Nuclear diameter statistics (pixels) ==========")
    stats_desc = diam_df["diameter_px"].describe(
        percentiles=[0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95]
    )
    print(stats_desc)
    print(f"\nTotal cells: {len(diam_df)}")
    print(f"Total nuclear transcripts: {diam_df['n_transcripts'].sum()}")
    print("=========================================================\n")


# ============================================================
# STEP 1_1 (simple 2D r_norm)
# ============================================================
def compute_simple_rnorm(nuclear_tx: pd.DataFrame) -> pd.DataFrame:
    print("Step 2: Computing 2D r_norm...")

    grouped = nuclear_tx.groupby("cell")
    centers = grouped[["x", "y"]].mean().rename(columns={"x": "cx", "y": "cy"})

    def calc_radius(sub):
        cx, cy = sub[["x", "y"]].mean().values
        dist = np.sqrt((sub["x"] - cx) ** 2 + (sub["y"] - cy) ** 2)
        return dist.max()

    centers["radius_2d"] = grouped.apply(calc_radius)

    def add_rnorm_2d(sub):
        cell_id = sub.name
        if cell_id not in centers.index:
            sub["r_norm_2d"] = np.nan
            return sub
        radius = float(centers.loc[cell_id, "radius_2d"])
        dist = np.sqrt((sub["x"] - centers.loc[cell_id, "cx"]) ** 2 +
                        (sub["y"] - centers.loc[cell_id, "cy"]) ** 2)
        sub["r_norm_2d"] = dist / (radius + 1e-12)
        return sub

    out = nuclear_tx.groupby("cell", group_keys=False).apply(add_rnorm_2d).copy()
    out["r_norm_2d"] = out["r_norm_2d"].clip(0.0, 1.0)
    return out


# ============================================================
# STEP 1_2 (3D correction / Wicksell)
# ============================================================
def wicksell_invert_cell_radius(
    cell_df: pd.DataFrame,
    diameter_col: str = DIAMETER_COL,
    max_diameter: float = None,
    sample_size: int = 40000,
    random_state: int = 42
) -> pd.DataFrame:

    print("Reading cell diameter data...")

    if DIAM_CELL_COL not in cell_df.columns:
        raise RuntimeError(f"Diameter table missing column: {DIAM_CELL_COL}")
    if diameter_col not in cell_df.columns:
        raise RuntimeError(f"Diameter table missing column: {diameter_col}")

    cell_df = cell_df.copy()
    cell_df[DIAM_CELL_COL] = cell_df[DIAM_CELL_COL].astype(str)
    cell_df[diameter_col] = pd.to_numeric(cell_df[diameter_col], errors="coerce")
    cell_df = cell_df.dropna(subset=[diameter_col]).copy()

    print(f"  Original number of cells: {len(cell_df)}")

    if max_diameter is not None:
        cell_df = cell_df[cell_df[diameter_col] <= float(max_diameter)].copy()
        print(f"  Cells after filtering: {len(cell_df)}")
    else:
        print("  No filtering applied (max_diameter = None)")

    cell_df["r_obs"] = cell_df[diameter_col] / 2.0

    if len(cell_df) > sample_size:
        cell_df = cell_df.sample(n=sample_size, random_state=random_state).copy()
        print(f"  Randomly sampled {sample_size} cells")

    r_obs = cell_df["r_obs"].to_numpy(dtype=float)
    if len(r_obs) < 10:
        raise RuntimeError("Too few cells for Wicksell inversion.")

    print("\nStarting Wicksell inversion (3D mode)...")
    print("Step 1/3: KDE estimation g(r)")
    kde = gaussian_kde(r_obs, bw_method="scott")

    r_grid = np.linspace(max(r_obs.min(), 1e-6), r_obs.max(), 300)
    g_r = kde(r_grid)

    print("Step 2/3: Numerical integration")
    R_grid = np.linspace(max(r_obs.min(), 1e-6), r_obs.max() * 1.5, 350)
    F_R = np.zeros_like(R_grid)

    for i, R in enumerate(tqdm(R_grid, desc="Abel integral")):
        mask = r_grid <= R
        r_vals = r_grid[mask]
        g_vals = g_r[mask]

        if len(r_vals) < 2:
            continue

        integrand = g_vals / np.sqrt(np.maximum(R**2 - r_vals**2, 1e-8))
        F_R[i] = np.trapz(integrand, r_vals)

    print("Step 3/3: Differentiation to recover f(R)")
    dF = np.gradient(F_R, R_grid)
    f_R = -2 * R_grid / np.pi * dF

    f_R[f_R < 0] = 0.0
    f_R = gaussian_filter1d(f_R, sigma=2)
    f_R /= (np.trapz(f_R, R_grid) + 1e-12)

    mu_R = np.trapz(R_grid * f_R, R_grid)
    sigma_R = np.sqrt(np.trapz((R_grid - mu_R) ** 2 * f_R, R_grid))
    print(f"Wicksell result: mu={mu_R:.2f}, sigma={sigma_R:.2f}")

    print("\nComputing R_est for each cell...")

    r_lookup = r_grid
    R_lookup = np.zeros_like(r_lookup)

    for i, r in enumerate(tqdm(r_lookup, desc="Lookup")):
        mask = R_grid >= r
        if np.sum(mask) < 5:
            R_lookup[i] = r
            continue

        weights = f_R[mask]
        R_vals = R_grid[mask]
        R_lookup[i] = np.sum(R_vals * weights) / (np.sum(weights) + 1e-12)

    interp_func = interp1d(r_lookup, R_lookup, fill_value="extrapolate", bounds_error=False)
    cell_df["R_est"] = interp_func(cell_df["r_obs"].values)
    cell_df["R_est"] = np.maximum(cell_df["R_est"], cell_df["r_obs"])

    print(f"R_est range: {cell_df['R_est'].min():.2f} ~ {cell_df['R_est'].max():.2f}")
    return cell_df


def apply_3d_correction(tx_df: pd.DataFrame, cell_df: pd.DataFrame) -> pd.DataFrame:
    print("\nLoading transcript data and applying 3D correction...")

    tx_df = tx_df.copy()
    tx_df["cell"] = tx_df["cell"].astype(str)
    cell_df = cell_df.copy()
    cell_df["cell"] = cell_df["cell"].astype(str)

    required_cols = {"cell", "r_obs", "R_est"}
    if not required_cols.issubset(cell_df.columns):
        raise RuntimeError(f"cell_df missing columns: {required_cols - set(cell_df.columns)}")

    tx_df = tx_df.merge(cell_df[["cell", "r_obs", "R_est"]], on="cell", how="inner")
    print(f"  Number of transcripts: {len(tx_df)}")

    ratio = tx_df["r_obs"].astype(float) / (tx_df["R_est"].astype(float) + 1e-12)
    r2d = tx_df["r_norm_2d"].astype(float).clip(0.0, 1.0)

    val = 1.0 - ratio**2 + (r2d * ratio) ** 2
    val = np.maximum(val, 0.0)

    tx_df["r_norm_3d"] = np.sqrt(val).clip(upper=1.0)
    return tx_df


# ============================================================
# STEP 1_3 MAIN
# ============================================================
def step1_compute_rnorm(tx_file: str, out_dir: str, mode: str = "2D") -> str:
    tx_df = load_transcripts(tx_file)
    nuclear_tx = get_nuclear_transcripts(tx_df)

    # Sample cells first, then all later steps use this subset
    sampled_nuclear_tx = sample_cells_first(
        nuclear_tx,
        sample_size=SAMPLE_SIZE,
        random_state=RANDOM_STATE
    )

    # Compute 2D r_norm on sampled subset
    sampled_nuclear_tx = compute_simple_rnorm(sampled_nuclear_tx)

    # Compute diameters only on sampled subset
    diam_df = compute_nuclear_diameters(sampled_nuclear_tx)
    diam_out = os.path.join(OUT_DESC, "nuclear_diameters.csv")
    diam_df.to_csv(diam_out, index=False)
    print(f"  Diameter data saved: {diam_out}")

    print_statistics(diam_df)
    plot_diameter_distribution(diam_df, OUT_DESC)

    mode = mode.upper().strip()
    if mode not in {"2D", "3D"}:
        raise ValueError('mode must be "2D" or "3D"')

    if mode == "2D":
        sampled_nuclear_tx["r_norm"] = sampled_nuclear_tx["r_norm_2d"]
        out_file = os.path.join(out_dir, "tx_nuclear_with_rnorm.csv")
        sampled_nuclear_tx.to_csv(out_file, index=False)
        print(f"  Saved 2D nuclear transcripts with r_norm: {out_file}")
        return out_file

    cell_df = wicksell_invert_cell_radius(
        diam_df,
        diameter_col=DIAMETER_COL,
        max_diameter=MAX_DIAMETER,
        sample_size=40000,
        random_state=RANDOM_STATE,
    )

    out_cell_file = os.path.join(out_dir, "cells_with_estimated_radius_sampled.csv")
    cell_df.to_csv(out_cell_file, index=False)
    print(f"  Saved cell radius estimation: {out_cell_file}")

    tx_3d = apply_3d_correction(sampled_nuclear_tx, cell_df)
    tx_3d["r_norm"] = tx_3d["r_norm_3d"]

    out_file = os.path.join(out_dir, "tx_nuclear_with_rnorm.csv")
    tx_3d.to_csv(out_file, index=False)
    print(f"  Saved 3D-corrected transcripts: {out_file}")

    print("\nComputing correlations...")
    r2d = tx_3d["r_norm_2d"].values
    r3d = tx_3d["r_norm_3d"].values

    pearson_r, _ = pearsonr(r2d, r3d)
    spearman_r, _ = spearmanr(r2d, r3d)

    print(f"Pearson r: {pearson_r:.4f}")
    print(f"Spearman r: {spearman_r:.4f}")

    max_points = 50000
    if len(r2d) > max_points:
        idx = np.random.choice(len(r2d), max_points, replace=False)
        x_plot = r2d[idx]
        y_plot = r3d[idx]
    else:
        x_plot = r2d
        y_plot = r3d

    plt.figure(figsize=(8, 8))
    plt.scatter(x_plot, y_plot, s=12, alpha=0.016)
    plt.plot([0, 1], [0, 1], "--", linewidth=1)
    plt.xlabel("2D r_norm")
    plt.ylabel("3D r_norm")
    plt.title(f"2D vs 3D\nPearson={pearson_r:.3f}, Spearman={spearman_r:.3f}")
    plt.xlim(0, 1)
    plt.ylim(0, 1)
    plt.tight_layout()

    plot_file = os.path.join(out_dir, "r_norm_2D_vs_3D_scatter.png")
    plt.savefig(plot_file, dpi=300)
    plt.close()
    print(f"  Plot saved: {plot_file}")

    return out_file


# ============================================================
# STEP 2: PDE model
# ============================================================
def normalize_r(r_norm, mode="center0_edge1"):
    r = np.asarray(r_norm, dtype=float)
    if mode == "edge0_center1":
        r = 1.0 - r
    return np.clip(r, 0.0, 1.0)


def pde_density_r(r, lmbda, beta):
    r = np.asarray(r, dtype=float)
    I0b = i0(1.0 / lmbda)
    I1b = i1(1.0 / lmbda)
    denom = (beta * I0b + (I1b / lmbda))
    C1 = -beta / denom
    c_r = 1.0 + C1 * i0(r / lmbda)
    return np.maximum(r * c_r, 1e-300)


def neg_loglik_pde(params, r):
    a, b = params
    lmbda = np.log1p(np.exp(a)) + 1e-6
    beta = np.log1p(np.exp(b)) + 1e-6

    grid = np.linspace(0.0, 1.0, 1001)
    f_grid = pde_density_r(grid, lmbda, beta)
    Z = np.trapz(f_grid, grid)

    f_r = pde_density_r(r, lmbda, beta) / (Z + 1e-12)
    ll = np.sum(np.log(f_r))

    reg = 1e-4 * (a * a + b * b)
    return -(ll - reg)


def fit_cell_pde(cell, r_values, min_tx=50, r_mode="center0_edge1"):
    if len(r_values) < min_tx:
        return None

    r = normalize_r(r_values, mode=r_mode)
    if np.std(r) < 1e-6:
        return None

    x0 = np.array([np.log(np.expm1(0.3)), np.log(np.expm1(1.0))])

    res = minimize(neg_loglik_pde, x0, args=(r,), method="L-BFGS-B")
    if not res.success:
        return None

    a, b = res.x
    lmbda = np.log1p(np.exp(a)) + 1e-6
    beta = np.log1p(np.exp(b)) + 1e-6

    grid = np.linspace(0.0, 1.0, 2001)
    f_grid = pde_density_r(grid, lmbda, beta)
    Z = np.trapz(f_grid, grid)
    pdf = f_grid / (Z + 1e-12)
    cdf = np.cumsum(pdf) * (grid[1] - grid[0])

    def model_cdf(x):
        x_arr = np.asarray(x, dtype=float)
        idx = np.searchsorted(grid, x_arr, side="right")
        idx = np.clip(idx, 0, len(cdf) - 1)
        return cdf[idx]

    ks_stat = kstest(r, model_cdf).statistic
    ll = -neg_loglik_pde(res.x, r)
    aic = 2 * 2 - 2 * ll

    return {
        "cell": str(cell),
        "lambda": float(lmbda),
        "beta": float(beta),
        "loglik": float(ll),
        "AIC": float(aic),
        "KS": float(ks_stat),
        "n_tx": int(len(r)),
        "mean_r": float(np.mean(r)),
        "median_r": float(np.median(r)),
    }


def step2_fit_all(tx_rnorm_file, out_dir, min_tx=50):
    print("Step 2: Loading r_norm file and fitting PDE per cell...")

    df = pd.read_csv(tx_rnorm_file)
    if "cell" not in df.columns or "r_norm" not in df.columns:
        raise RuntimeError("Input file missing required columns ('cell','r_norm').")

    cell_groups = df.groupby("cell")["r_norm"].apply(list)
    print(f"  Found {len(cell_groups)} cells with r_norm values.")

    rand_cell = np.random.choice(cell_groups.index, 1)[0]
    example_res = fit_cell_pde(rand_cell, cell_groups[rand_cell], min_tx=min_tx)

    if example_res:
        r = normalize_r(cell_groups[rand_cell])
        lmbda, beta = example_res["lambda"], example_res["beta"]
        grid = np.linspace(0, 1, 200)

        f_grid = pde_density_r(grid, lmbda, beta)
        f_grid /= (np.trapz(f_grid, grid) + 1e-12)

        plt.figure(figsize=(6, 4))
        plt.hist(r, bins=20, range=(0, 1), density=True, alpha=0.5, label="Empirical")
        plt.plot(grid, f_grid, "-", lw=2, label="PDE fit")
        plt.xlabel("r (normalized distance)")
        plt.ylabel("Density")
        plt.title(f"Example Cell {rand_cell}\nλ={lmbda:.3f}, β={beta:.3f}, KS={example_res['KS']:.3f}")
        plt.legend()
        example_fig = os.path.join(out_dir, f"fit_example_{rand_cell}.png")
        plt.tight_layout()
        plt.savefig(example_fig, dpi=300)
        plt.close()
        print(f"  Example fit saved: {example_fig}")

    results = []
    total = len(cell_groups.index)

    for i, cell in enumerate(cell_groups.index, start=1):
        res = fit_cell_pde(cell, cell_groups[cell], min_tx=min_tx)
        if res:
            results.append(res)
        if i % 500 == 0 or i == total:
            sys.stdout.write(f"\r  Progress: {i}/{total} cells ({i/total*100:.1f}%)")
            sys.stdout.flush()

    print("\n  Fitting finished.")
    results_df = pd.DataFrame(results)
    out_csv = os.path.join(out_dir, "fit_results_all_cells.csv")
    results_df.to_csv(out_csv, index=False)
    print(f"  Saved fit results: {out_csv}")

    if len(results_df) > 0:
        plt.figure(figsize=(12, 4))
        plt.subplot(1, 3, 1)
        plt.hist(results_df["lambda"], bins=50)
        plt.xlabel("lambda (λ)")
        plt.title("Distribution of λ")

        plt.subplot(1, 3, 2)
        plt.hist(results_df["beta"], bins=50)
        plt.xlabel("beta (β)")
        plt.title("Distribution of β")

        plt.subplot(1, 3, 3)
        plt.hist(results_df["KS"], bins=50)
        plt.xlabel("KS statistic")
        plt.title("KS (fit quality)")

        plt.tight_layout()
        fig_out = os.path.join(out_dir, "fit_params_distribution.png")
        plt.savefig(fig_out, dpi=300)
        plt.close()
        print(f"  Parameter distribution plot saved: {fig_out}")
    else:
        print("  No successful fits to plot.")

    return out_csv


# ============================================================
# STEP 3: Spatial KS map
# ============================================================
def step3_plot_ks_map(fit_csv, meta_file, out_dir,
                      vmin_percentile=PLOT_VMIN_PERCENTILE,
                      vmax_percentile=PLOT_VMAX_PERCENTILE):
    if meta_file is None or not os.path.exists(meta_file):
        print("Step 3: metadata file not found — skipping spatial KS map.")
        return None

    print("Step 3: Plotting spatial KS map...")
    fit_df = pd.read_csv(fit_csv)
    meta_df = pd.read_csv(meta_file)

    fit_df["cell"] = fit_df["cell"].astype(str)
    meta_df = standardize_metadata(meta_df)

    merged = pd.merge(
        fit_df[["cell", "KS"]],
        meta_df[["cell", "meta_x", "meta_y"]],
        on="cell",
        how="inner",
    )

    if merged.empty:
        print("  No matching cell IDs found — skipping.")
        return None

    vmin, vmax = get_percentile_limits(
        merged["KS"],
        low_pct=vmin_percentile,
        high_pct=vmax_percentile
    )
    merged["KS_clipped"] = merged["KS"].clip(vmin, vmax)

    print(f"  KS color limits: vmin={vmin:.4f}, vmax={vmax:.4f} "
          f"({vmin_percentile}%-{vmax_percentile}%)")

    plt.figure(figsize=(10, 10))
    sns.set(style="white", font_scale=1.1)

    sc = plt.scatter(
        merged["meta_x"],
        merged["meta_y"],
        c=merged["KS_clipped"],
        cmap="YlOrRd",
        s=20,
        edgecolors="none",
        vmin=vmin,
        vmax=vmax,
    )
    cbar = plt.colorbar(sc, fraction=0.046, pad=0.04)
    cbar.set_label("KS (fit reliability)")
    plt.xlabel("X coordinate (px)")
    plt.ylabel("Y coordinate (px)")
    plt.title("Spatial KS Reliability Map")
    plt.gca().invert_yaxis()

    out_file = os.path.join(out_dir, "spatial_KS_map.png")
    plt.savefig(out_file, dpi=600, bbox_inches="tight")
    plt.close()
    print(f"  Spatial KS map saved: {out_file}")
    return out_file


# ============================================================
# STEP 4: Export probability and delta
# ============================================================
def s_flux_r(r, lmbda, beta):
    I0b = i0(1.0 / lmbda)
    I1b = i1(1.0 / lmbda)
    C1 = -beta / (beta * I0b + I1b / lmbda)
    c_r = 1.0 + C1 * i0(r / lmbda)
    s_r = np.maximum(r * c_r, 1e-10)
    return s_r


def step4_compute_delta_and_future(tx_rnorm_file, fit_csv, base_dir, out_dir):
    print("Step 4: Computing per-transcript export probabilities and building matrices...")

    tx_df = pd.read_csv(tx_rnorm_file)
    fit_df = pd.read_csv(fit_csv)

    # IMPORTANT: tx_rnorm_file may already be standardized, so this now works safely
    tx_df = standardize_transcripts(tx_df, require_gene=True)
    tx_df["cell"] = tx_df["cell"].astype(str)
    fit_df["cell"] = fit_df["cell"].astype(str)

    all_results = []
    fit_cells = set(fit_df["cell"].unique())
    grouped = tx_df.groupby("cell")

    for cell, group in grouped:
        if cell not in fit_cells:
            continue

        row = fit_df.loc[fit_df["cell"] == cell].iloc[0]
        lmbda = float(row["lambda"])
        beta = float(row["beta"])

        rvals = normalize_r(group["r_norm_2d"].values)
        s_vals = s_flux_r(rvals, lmbda, beta)

        alpha = beta * len(rvals) / (np.sum(s_vals) + 1e-10)
        p_export = 1 - np.exp(-alpha * s_vals)

        tmp = pd.DataFrame({
            "cell": [cell] * len(p_export),
            "gene": group["gene"].values,
            "p_export": p_export
        })
        all_results.append(tmp)

    if len(all_results) == 0:
        raise RuntimeError("No export probabilities computed.")

    all_df = pd.concat(all_results, axis=0)
    print(f"  Combined per-transcript export entries: {len(all_df)}")

    delta_export_prob = all_df.groupby(["cell", "gene"])["p_export"].sum().reset_index()

    cells = delta_export_prob["cell"].astype(str).unique()
    genes = delta_export_prob["gene"].astype(str).unique()
    delta_mat = pd.DataFrame(0.0, index=cells, columns=genes)

    for row in delta_export_prob.itertuples(index=False):
        delta_mat.at[str(row.cell), str(row.gene)] = float(row.p_export)

    delta_out = os.path.join(out_dir, "delta_export_prob.csv")
    delta_mat.to_csv(delta_out)
    print(f"  Saved delta matrix: {delta_out}")

    delta_median = delta_mat.median(axis=0)
    delta_standardized = delta_mat.subtract(delta_median, axis=1)
    delta_med_out = os.path.join(out_dir, "delta_export_prob_median.csv")
    delta_standardized.to_csv(delta_med_out)
    print(f"  Saved median-standardized delta: {delta_med_out}")

    all_tx = pd.read_csv(TX_IN_FILE, dtype=str)
    all_tx = standardize_transcripts(all_tx, require_gene=True)

    df_cyto_trans = all_tx[all_tx["CellComp"].astype(str).str.lower() == CYTOPLASM_LABEL].copy()
    cyto_mat = pd.crosstab(df_cyto_trans["cell"], df_cyto_trans["gene"])
    cyto_mat.index = cyto_mat.index.astype(str)

    print(f"  Cytoplasm counts shape: {cyto_mat.shape}")

    common_cells = cyto_mat.index.intersection(delta_mat.index)
    common_genes = cyto_mat.columns.intersection(delta_mat.columns)
    print(f"  Intersection: {len(common_cells)} cells, {len(common_genes)} genes")

    cyto_mat_common = cyto_mat.loc[common_cells, common_genes]
    delta_common = delta_mat.loc[common_cells, common_genes]

    cyto_out = os.path.join(base_dir, "transcripts_cytoplasm_matrix_common.csv")
    cyto_mat_common.to_csv(cyto_out)
    print(f"  Saved cytoplasm matrix (common): {cyto_out}")

    future_matrix = cyto_mat_common.add(delta_common, fill_value=0.0)
    future_out = os.path.join(out_dir, "future.csv")
    future_matrix.to_csv(future_out)
    print(f"  Saved future matrix: {future_out}")

    return delta_out, delta_med_out, future_out


# ============================================================
# MAIN
# ============================================================
def run_pipeline():
    tx_rnorm_file = step1_compute_rnorm(TX_IN_FILE, OUT_PART1, mode=NORM_MODE)
    fit_csv = step2_fit_all(tx_rnorm_file, OUT_PART2, min_tx=50)

    try:
        ks_map = step3_plot_ks_map(fit_csv, META_FILE, OUT_PART2)
    except Exception as e:
        print(f"Step 3 skipped or failed: {e}")

    delta_out, delta_med_out, future_out = step4_compute_delta_and_future(
        tx_rnorm_file, fit_csv, BASE_DIR, OUT_PART3
    )

    print("\nPipeline finished.")
    print(
        f"Outputs:\n"
        f"  Step 1 (rnorm file): {tx_rnorm_file}\n"
        f"  Step 2 (fit results): {fit_csv}\n"
        f"  Step 3 (KS map): {ks_map if 'ks_map' in locals() else 'skipped'}\n"
        f"  Step 4 (delta): {delta_out}\n"
        f"  Step 4 (delta_median): {delta_med_out}\n"
        f"  Step 4 (future): {future_out}"
    )


if __name__ == "__main__":
    run_pipeline()